In [8]:
import torch
from torch import nn
import numpy as np

In [108]:
class BengioLM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, context_size = 4):
        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim
        self.context_size = context_size
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.W = nn.Linear(context_size * embedding_dim, hidden_dim)
        self.tanh = nn.Tanh()
        self.U = nn.Linear(hidden_dim, vocab_size)
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, input_token_ids):
        e = self.embedding(input_token_ids)
        e = e.view(e.size(0),-1)
        h = self.W(e)
        h = self.tanh(h)
        logits = self.U(h)
        probabilities = self.softmax(logits)
        log_probabilities = nn.LogSoftmax(dim=-1)(logits)
        return log_probabilities


In [96]:
class Vocabulary:
    def __init__(self):
        self._token_by_id = {} #id => token
        self._id_by_token = {} #token => id

    def add_token(self, token: str):
        if token in self._id_by_token:
            return
        next_id = len(self._token_by_id)
        self._token_by_id[next_id] = token
        self._id_by_token[token] = next_id
    @classmethod
    def from_file(self, path):
        vocab = Vocabulary()
        with open(path, encoding="utf-8") as file:
            for line in file:
                tokens = line.split()
                for token in tokens:
                    vocab.add_token(token)
        return vocab


    def token_by_id(self, token_id):
        return self._token_by_id[token_id]
    def id_by_token(self, token):
        return self._id_by_token[token]
    def __len__(self):
        return len(self._token_by_id)

In [82]:
from typing import List, Tuple

def tokenize(line: str, vocab) -> List[int]:
    """Повертає індекси токенів з тексту"""
    tokens = line.split()
    indexes = [vocab.id_by_token(token) for token in tokens]
    return indexes


def prepare_train_examples(dataset, vocab, context_size) -> List[Tuple[List[int], int]]:
    result = []
    for line in dataset:
        token_ids = tokenize(line, vocab)

        for i in range(len(token_ids) - context_size - 1):
            context = token_ids[i: i + context_size]
            target = token_ids[i + context_size]
            result.append((context, target))

    return result


In [84]:
vocab = Vocabulary.from_file("train.bpe.txt")
print(vocab._token_by_id)
print(vocab.id_by_token("спить"))
print(vocab.token_by_id(1))
dataset = open("train.bpe.txt", encoding="utf-8").readlines()
train_examples = prepare_train_examples(dataset, vocab, context_size=2)
print(train_examples)

{0: 'Кіт', 1: 'спить', 2: 'на', 3: 'кухн@@', 4: 'і', 5: 'Собака', 6: 'біг@@', 7: 'ав', 8: 'у', 9: 'саду', 10: 'Птах', 11: 'лет@@', 12: 'ить', 13: 'над', 14: 'б@@', 15: 'удин@@', 16: 'ком', 17: 'Кролик', 18: 'хова@@', 19: 'в@@', 20: 'ся', 21: 'під', 22: 'сто@@', 23: 'л@@', 24: 'ом', 25: 'Кінь', 26: 'іде', 27: 'п@@', 28: 'о', 29: 'пол@@', 30: 'ю', 31: 'Корова', 32: 'я@@', 33: 'ла', 34: 'біля', 35: 'са@@', 36: 'ра@@', 37: 'сидить', 38: 'ст@@', 39: 'і@@', 40: 'ь@@', 41: 'ц@@', 42: 'сп@@', 43: 'дв@@', 44: 'ере@@', 45: 'й', 46: 'К@@', 47: 'а@@', 48: 'чка', 49: 'ва@@', 50: 'є', 51: 'ста@@', 52: 'ку', 53: 'Л@@', 54: 'исиця', 55: 'через', 56: 'лі@@', 57: 'с', 58: 'М@@', 59: 'иша', 60: 'ховається', 61: 'за', 62: 'ко@@', 63: 'ро@@', 64: 'кою', 65: 'В@@', 66: 'івця', 67: 'ходила', 68: 'д@@', 69: 'оро@@', 70: 'г@@', 71: 'урка', 72: 'ї@@', 73: 'с@@', 74: 'ть', 75: 'дворі', 76: 'навколо', 77: 'по@@', 78: 'ля', 79: 'ч@@', 80: 'е@@', 81: 'к@@', 82: 'в', 83: 'р@@', 84: 'и@@', 85: 'орі', 86: 'ж@@', 87: '

In [142]:
CONTEXT_SIZE = 2
def train():
    vocab = Vocabulary.from_file("train.bpe.txt")
    model = BengioLM(vocab_size = len(vocab), embedding_dim=8, hidden_dim=6, context_size = CONTEXT_SIZE)

    optimizer = torch.optim.SGD(model.parameters(), 0.1)

    for epoch in range(50):
        epoch_loss = 0.0
        for context, target in train_examples:
            context = torch.tensor(context).unsqueeze(0)
            log_probs = model(context)
            loss = -log_probs.squeeze(0)[target] ##negative log likelihood loss

            #Виконання кроку Back-Propagation
            epoch_loss += loss.item()
            loss.backward()

            optimizer.step()
            optimizer.zero_grad()
        average_loss = epoch_loss / len(train_examples)
        print(f"Epoch {epoch + 1}, Average Loss: {average_loss:.4f}")



In [143]:
train()

Epoch 1, Average Loss: 4.6353
Epoch 2, Average Loss: 4.2032
Epoch 3, Average Loss: 3.8245
Epoch 4, Average Loss: 3.4731
Epoch 5, Average Loss: 3.1621
Epoch 6, Average Loss: 2.8930
Epoch 7, Average Loss: 2.6610
Epoch 8, Average Loss: 2.4633
Epoch 9, Average Loss: 2.2952
Epoch 10, Average Loss: 2.1482
Epoch 11, Average Loss: 2.0243
Epoch 12, Average Loss: 1.9201
Epoch 13, Average Loss: 1.8361
Epoch 14, Average Loss: 1.7613
Epoch 15, Average Loss: 1.7006
Epoch 16, Average Loss: 1.6472
Epoch 17, Average Loss: 1.5919
Epoch 18, Average Loss: 1.5586
Epoch 19, Average Loss: 1.5089
Epoch 20, Average Loss: 1.4597
Epoch 21, Average Loss: 1.4204
Epoch 22, Average Loss: 1.3719
Epoch 23, Average Loss: 1.3318
Epoch 24, Average Loss: 1.3046
Epoch 25, Average Loss: 1.2673
Epoch 26, Average Loss: 1.2499
Epoch 27, Average Loss: 1.2207
Epoch 28, Average Loss: 1.1954
Epoch 29, Average Loss: 1.1737
Epoch 30, Average Loss: 1.1672
Epoch 31, Average Loss: 1.1524
Epoch 32, Average Loss: 1.1293
Epoch 33, Average